# Нефтекод 2026 — мультиагентная система управления производством дизеля

Ноутбук запускает весь пайплайн на данных организаторов: от сырых файлов до карточки оператора.

**Что делает система.** Раз в час читает состояние установки гидроочистки 24-2000, оценивает качество
и надёжность, перебирает шаги управляющих параметров, отсекает всё, что нарушает жёсткие ограничения,
и выдаёт оператору одно из трёх: **изменить режим**, **не менять** или **отказаться** с причиной.
Затем подбирает товарную смесь из трёх резервуаров под спецификацию.

| Агент | Что делает |
|---|---|
| Агент качества | сера гидроочищенного ДТ: уровень, интервал p05–p95, вероятность нарушения |
| Агент надёжности | тяжесть режима, модельный диапазон рычагов, ресурс катализатора |
| Оптимизатор | шаги рычагов → жёсткий фильтр → ранжирование |
| Агент блендинга | доли компонентов и доза присадки под спецификацию; **серный бюджет** для гидроочистки |
| Оркестратор | решение RECOMMEND / HOLD / REFUSE и трасса решения |

Агенты не просто вызываются по очереди. Агент блендинга до решения сообщает гидроочистке **серный
бюджет** — сколько серы в гидроочищенном ДТ резервуар доведёт до нормы, — и жёсткая проверка режима идёт
уже по нему: норма 10 мг/кг по ТЗ относится к товарной смеси, а не к промежуточному продукту.

Ни один агент не вызывает LLM и не читает файлы: всё приходит через состояние, поэтому одинаковый
вход даёт одинаковый результат, а система работает без интернета.

**Как запустить**

1. `pip install -r requirements.txt`, плюс `pyarrow` и `jupyter`.
2. Положить пакет организаторов в `data/raw/`: `242000_tags.csv`, `ЛИМСы … .xlsx`, `Выгрузка ПАК … .xlsx`.
3. Выполнить все ячейки. С нуля — около полутора минут, включая сборку кэша из сырых файлов.

Параметры сценариев — в ячейке 2, их можно менять.

In [1]:
import contextlib
import io
import json
import os
import sys
import time
from pathlib import Path

T0 = time.time()
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
for path in (ROOT, ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import pandas as pd

from scripts import build_cache
from scripts import realdata as rd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("репозиторий:", ROOT.name)
print("python", sys.version.split()[0], "| pandas", pd.__version__)
print()
# Кэш parquet строится из сырых файлов один раз; дальше всё читает только его.
for name, what in build_cache.build().items():
    print(f"{name:<12} {what}")

репозиторий: Neftecode
python 3.13.1 | pandas 3.0.5

tags_242000  already built
lims_long    already built
pak_long     already built


In [2]:
# ── Параметры, которые можно менять ────────────────────────────────────────────────
# Организаторы: «сценарии не должны быть жёсткими — результаты должны меняться
# в зависимости как от качества сырья, так и от задания на качество продукции».

RETRAIN = True            # False — взять готовые models/*.json, если они уже есть
EVERY_MINUTES = 60        # шаг решений в демо-неделях; организаторы просят 15–60 мин

SCENARIO_AT = "2026-01-27T16:00"   # момент решения для сценариев, отложенный период
FEED_SULFUR_PCT = 1.25             # сера в сырье гидроочистки, % масс.; измерено ≈ 1.03
SEASON = "summer"                  # summer | winter — нормы плотности и цетанового числа
STOCKS_T = {}                      # запасы резервуаров, т, например {"hydrotreated_diesel": 300}

## 1. Данные: что в них на самом деле

- **Пропуски закодированы значениями-заглушками** — `307`, `0` и `10` на 24-2000; `NaN` в файлах нет вообще.
  Заглушки убираются первым шагом, до любых расчётов.
- **Справочник тегов 24-2000 в исходном пакете был перепутан.** 16.09 организаторы прислали исправленный —
  рычаги и единицы ниже взяты из него, и каждый проверен по данным. Четыре рычага совпадают с четырьмя
  управляемыми параметрами на схеме гидроочистки организаторов: расход, давление, отношение ВСГ/сырьё, температура.
- **Анализ ЛИМС доступен через 4 ч после отбора** — подтверждено организаторами. Модель ни разу не видит
  результат раньше, чем его увидел бы оператор.
- **Остановы записаны почти-нулями**, а не заглушками, — их ловит отдельное правило «установка работает».

In [3]:
raw = rd.read_cache("tags_242000", columns=["date", *rd.LEVERS])
clean = rd.load_telemetry()
display(pd.DataFrame([
    {
        "тег": rd.tag_id(tag),
        "что это (справочник 16.09)": spec["label"],
        "единица": spec["unit"],
        "шаг за цикл": spec["step"],
        "заглушек, %": round(100 * raw[tag].isin(rd.SENTINELS_242000).mean(), 2),
    }
    for tag, spec in rd.LEVERS.items()
]))

running = rd.running_state(clean)
print(f"телеметрия: {len(clean):,} строк с шагом 10 мин, "
      f"{clean.index.min():%Y-%m-%d} → {clean.index.max():%Y-%m-%d}")
print(f"установка работает {(running == True).mean():.1%} времени, стоит {(running == False).mean():.1%}")
print(f"разбиение по времени, без перемешивания: обучение до {rd.SPLIT_AT:%Y-%m-%d %H:%M}, "
      f"дальше — отложенный период")
print(f"задержка публикации анализа ЛИМС: {rd.LIMS_DELAY}")

,тег,что это (справочник 16.09),единица,шаг за цикл,"заглушек, %"
0,242000:T5,Полисеп. Р-201. Температура ГСС на выходе,°C,2.00,0.00
1,242000:F26,"Расход гидроочищенного ДТ в цех №8, объёмный",м³/ч,5.00,0.21
2,242000:F2,Газовая схема. Расход газа на линии от ЦК-201,нм³/ч,1000.00,2.88
3,242000:P13,Полисеп. Р-202. Давление на входе,МПа,0.05,0.00


телеметрия: 189,217 строк с шагом 10 мин, 2023-01-01 → 2026-08-07
установка работает 95.2% времени, стоит 4.6%
разбиение по времени, без перемешивания: обучение до 2025-09-12 14:30, дальше — отложенный период
задержка публикации анализа ЛИМС: 0 days 04:00:00


## 2. Метки: лабораторная сера

Метка — сера по ЛИМС в гидроочищенном ДТ (точка отбора 2), только пока установка работает.
Три результата выше 50 мг/кг помечены как выбросы: все пришлись на первые часы после пуска установки.

In [4]:
from scripts.analysis import (
    blend_components, breach_classifier, labels, predictability,
    quality_baseline_fit, reliability_reference,
)

def fitted(name: str) -> bool:
    return (rd.MODELS_DIR / name).exists()

def model(name: str) -> dict:
    return json.loads((rd.MODELS_DIR / name).read_text(encoding="utf-8"))

if RETRAIN or not fitted("labels.parquet"):
    labels.main()

wrote models/labels.parquet
lab results            : 1,462
running at sampling    : 1,462
not running            : 0
running unknown (gaps) : 0

            n  median  above_spec  outliers
split                                      
heldout   395     8.8          66         0
train    1067     8.5         164         3

outliers above 50 mg/kg:
         sampled_at  sulfur_mg_kg  running_at_sample  hours_since_restart  feed_sulfur_pct_same_day  feed_as_mg_kg
2024-04-23 06:30:00        2120.0               True                 60.3                       NaN            NaN
2024-04-23 10:00:00         107.0               True                 63.8                       NaN            NaN
2025-07-24 13:00:00         120.0               True                  2.7                       NaN            NaN


## 3. Агент качества: базовая модель

Соседние анализы почти не связаны, поэтому прогноз — экспоненциальное сглаживание доступных анализов, а не
«как вчера». Интервал p05–p95 — квантили ошибки этого прогноза. Всё подобрано только на обучающем периоде.

In [5]:
if RETRAIN or not fitted("quality_baseline.json"):
    with contextlib.redirect_stdout(io.StringIO()):
        quality_baseline_fit.main()
q = model("quality_baseline.json")
print(f"сглаживание α = {q['alpha']}, интервал ошибки {q['resid_q05']} … +{q['resid_q95']} мг/кг")
display(pd.Series(q["heldout_eval"], name="отложенный период").to_frame())

сглаживание α = 0.1, интервал ошибки -3.4721 … +3.095 мг/кг


,отложенный период
n_results,395
mae_ewma,1.4642
mae_train_median,1.5177
r2_ewma,0.01
r2_train_median,-0.0007
interval_coverage_p05_p95,0.9038
share_above_spec,0.1671
share_p95_below_spec,0.0557
p95_range,"[8.369, 13.02]"


## 4. Предсказуема ли сера по телеметрии? Честный отрицательный результат

Ридж на средних по окнам 2/6/24 ч, лаги 0–24 ч; выбор только по кросс-валидации внутри обучения.
**R² на отложенном периоде около нуля: уровень серы по телеметрии не предсказывается.** Модель, которая не лучше
сглаживания, мы за прогноз не выдаём.

Вероятная причина названа самими организаторами: на установке стоят системы регулирования с обратной связью,
которые быстро компенсируют изменения качества, — поэтому в истории связь «рычаг → сера» замаскирована.

In [6]:
if RETRAIN or not fitted("predictability.json"):
    with contextlib.redirect_stdout(io.StringIO()):
        predictability.main()
pr = model("predictability.json")
sel = pr["selected_by_train_cv"]
print(f"выбрано по CV обучения: {sel['variant']}, лаг {sel['lag_h']} ч")
print(f"  R² на отложенном {sel['r2_heldout']:.3f} | MAE {sel['mae_heldout']:.3f} "
      f"против {sel['mae_ewma']:.3f} у сглаживания")

выбрано по CV обучения: levers+context+pak, лаг 0 ч
  R² на отложенном -0.005 | MAE 1.271 против 1.328 у сглаживания


## 5. Классификатор «анализ выше 10 мг/кг»

Уровень не предсказывается — а хвост? Критерии приёмки заданы **до** запуска: AUC ≥ 0.60, нижняя граница
бутстрапа выше 0.5, Brier skill > 0, лучше текущей оценки агента. Отбор — только по CV обучения, отложенный
период оценён один раз.

Итог: сигнал есть, но **только на горизонте 0–2 ч**, и несёт его поточный анализатор, который стоит в той же
точке отбора, что лаборатория. Это оценка «сейчас», а не прогноз — и ровно тот горизонт 0–3 ч, который
просят организаторы. Если анализатор заморожен, классификатор отключается сам.

In [7]:
if RETRAIN or not fitted("breach_classifier.json"):
    with contextlib.redirect_stdout(io.StringIO()):
        breach_classifier.main()
bc = model("breach_classifier.json")
print("отложенный период:", bc["heldout"])
print("AUC, бутстрап 95 %:", bc["auc_bootstrap_ci95"])
display(pd.Series(bc["acceptance_checks"], name="критерий выполнен").to_frame())

decay = pd.DataFrame(bc["feature_group_ablation"]).pivot(index="group", columns="horizon_h", values="auc")
decay.columns = [f"{h} ч" for h in decay.columns]
print("AUC на отложенном по группам признаков и горизонту — сигнал живёт два часа:")
display(decay.round(3))

отложенный период: {'auc': 0.724, 'pr_auc': 0.3525, 'brier': 0.13759, 'brier_skill': 0.046}
AUC, бутстрап 95 %: [0.6601, 0.7248, 0.7844]


,критерий выполнен
auc_at_least_0.60,True
bootstrap_lower_above_0.50,True
brier_skill_positive,True
beats_quality_agent_today,True


AUC на отложенном по группам признаков и горизонту — сигнал живёт два часа:


,0 ч,2 ч,4 ч,6 ч,8 ч,12 ч,24 ч
group,,,,,,,
context,0.641,0.463,0.411,0.385,0.419,0.427,0.387
context+pak,0.673,0.475,0.442,0.428,0.421,0.441,0.397
levers,0.628,0.556,0.480,0.470,0.415,0.416,0.469
levers+context,0.704,0.541,0.484,0.425,0.487,0.494,0.484
levers+context+pak,0.707,0.543,0.492,0.449,0.495,0.515,0.488
pak,0.676,0.569,0.538,0.531,0.557,0.524,0.538


## 6. Агент надёжности

Рабочий диапазон рычагов — p01–p99 обучающего периода. **Это модельная граница, не заводской лимит**:
паспортных диапазонов в пакете нет. Текущий режим допустим с запасом в один шаг за границей — иначе решение
дребезжит, когда установка работает у края. Тяжесть режима — насколько реактор горячее обычного при той же
производительности, плюс загрузка и размер шага.

**Ресурс катализатора.** Избыток температуры реактора при той же загрузке растёт примерно на 1 °C в месяц и
сбрасывается при замене катализатора. Замены находятся по данным, а уровень замены берётся из обучающего
периода — это наблюдаемая практика завода, а не паспортная граница.

In [8]:
if RETRAIN or not fitted("reliability_reference.json"):
    with contextlib.redirect_stdout(io.StringIO()):
        reliability_reference.main()
ref = model("reliability_reference.json")
display(pd.DataFrame(ref["envelope"]).T.rename(
    columns={"p01": "нижняя граница (p01)", "p50": "медиана", "p99": "верхняя граница (p99)"}
))
cat = ref["catalyst"]
print(f"замены катализатора в обучающем периоде: {cat['changes_in_training']}")
print(f"уровень замены — избыток T5 перед заменой: {cat['eor_excess_c']:+.2f} °C")

,нижняя граница (p01),медиана,верхняя граница (p99)
242000:T5,347.0308,368.9160,387.8939
242000:F26,155.4412,255.3714,302.1268
242000:F2,80465.1824,93505.2930,106149.2094
242000:P13,3.6134,3.9176,4.0332


замены катализатора в обучающем периоде: ['2024-04-17T14:50:00']
уровень замены — избыток T5 перед заменой: +13.56 °C


## 7. Компоненты товарной смеси

Очищенный ДТ описан лабораторией полностью. Керосин и газойль — фракциями АВТ, приведёнными к
гидроочищенному виду; их цетановое число рассчитано по **ASTM D976** и проверено на 37 парах замеров
гидроочищенного ДТ. Выдумана только сера керосина и газойля — анализов по ним нет.

In [9]:
if RETRAIN or not fitted("blend_components.json"):
    with contextlib.redirect_stdout(io.StringIO()):
        blend_components.main()
import yaml
from neftecode.agents.blending import build_components

comp = model("blend_components.json")
print("ASTM D976 на гидроочищенном ДТ:", comp["cetane_index_d976_validation"])
config = yaml.safe_load((ROOT / "configs" / "blending.yaml").read_text(encoding="utf-8"))
display(pd.DataFrame([
    {
        "компонент": c.label,
        "сера, мг/кг": c.sulfur_mg_kg,
        "плотность, кг/м³": c.density_kg_m3,
        "T95, °C": c.t95_c,
        "цетановое число": c.cetane,
        "запас, т": c.stock_t,
        "откуда сера": c.sources.get("sulfur_mg_kg", ""),
        "откуда цетан": c.sources.get("cetane", ""),
    }
    for c in build_components(comp, config)
]).set_index("компонент"))

ASTM D976 на гидроочищенном ДТ: {'n_pairs': 37, 'bias': -1.136, 'mae': 1.71, 'max_abs_error': 5.417, 'measured_median': 53.8, 'computed_median': 53.31}


,"сера, мг/кг","плотность, кг/м³","T95, °C",цетановое число,"запас, т",откуда сера,откуда цетан
компонент,,,,,,,
"Очищенный дизель (гидроочистка, т.о. 2)",8.6,836.10,347.0,53.75,1200.0,"lims: Гидроочистка т.о. 2, Mg.Sulfur","lims: Гидроочистка т.о. 2, CetaneNumber"
"Керосин (прокси: АВТ т.о. 2, гидроочищенный)",5.0,821.45,290.0,52.13,300.0,"допущение, configs/blending.yaml","ASTM D976 по плотности и T50, формула проверен..."
"Газойль (прокси: АВТ т.о. 1, гидроочищенный)",8.0,870.55,349.0,48.51,400.0,"допущение, configs/blending.yaml","ASTM D976 по плотности и T50, формула проверен..."


## 8. Один цикл решения — карточка оператора

Блоки, которых требует ТЗ: состояние, проблема, действие, ожидаемый эффект, проверенные ограничения,
уверенность, объяснение — плюс блок товарной смеси. Анализатор показан отдельно от прогноза: это признак,
а не результат анализа.

Под картой — то, что агенты сообщили друг другу: **серный бюджет** резервуара (в проверках видно, что лимит
серы для режима пришёл от агента смеси), **ресурс катализатора** и **резерв экономии** — насколько можно
охладить реактор, оставаясь в бюджете. Последнее — информация для технолога, а не рекомендация.

In [10]:
from scripts import run_real

_ = run_real.main(["--at", "2026-07-16T14:00"])

═══ 2026-07-16 14:00 · RECOMMEND ═══
1 Состояние   242000:T5 373.0 · 242000:F26 299.8 · 242000:F2 94 981 · 242000:P13 3.97
              ЛИМС 11.0 мг/кг (4 ч назад) · сглаженный уровень 8.05 мг/кг
              ПАК 9.2 ppm (исправен) — признак, не результат анализа
2 Проблема    последний анализ ЛИМС вне спецификации: 11.0 > 10 мг/кг; вероятность нарушения спецификации 30% (порог 25%, классификатор)
3 Действие    242000:T5 373.0 → 375.0
4 Эффект      сера 8.05 → 7.42 мг/кг, p95 11.15 → 10.51 · риск нарушения 21% (классификатор) · тяжесть medium (0.42)
5 Проверки    sulfur_mg_kg <= 11.429; controllable tags within configured ranges; blend shares sum == 1.0 (when blend tags present); reliability.is_mode_allowed; товарная смесь: сера ≤ 10 мг/кг; товарная смесь: T95 ≤ 360 °C; товарная смесь: цетановое число ≥ 51 (лето); товарная смесь: плотность 820–845 кг/м³ (лето); товарная смесь: доли в сумме 100 %, в пределах запасов
6 Уверенность 0.76
7 Почему      Рекомендация: 242000:T5: 373.0 → 375

## 9. Три демонстрационные недели

Все недели — из отложенного периода, модели их не видели. Решение принимается каждый час.

| Неделя | Что должна сделать система |
|---|---|
| устойчивая, с 25.01.2026 | **не вмешиваться** |
| риск по качеству, с 12.07.2026 | **действовать** |
| деградация данных, с 02.07.2026 | установка **работает**, а данные отказывают: **отказываться**, пока нет ни свежего анализа, ни исправного анализатора; **решать на запасной оценке**, пока не работает только анализатор |
| останов, с 21.06.2026 | **отказаться** — установка не работает |

Отказ — оцениваемая часть решения: система говорит, почему не может дать рекомендацию, и называет
причину конкретно — какой анализ устарел или какой рычаг вне диапазона.

In [11]:
shares = {}
for week in ("stable", "risk", "degraded", "shutdown"):
    recs = run_real.main(["--week", week, "--every", str(EVERY_MINUTES)])
    counts = pd.Series([run_real.outcome(r) for r in recs]).value_counts()
    shares[week] = (100 * counts / len(recs)).round(1)
    print("\n" + "─" * 110 + "\n")

summary = pd.DataFrame(shares).T.fillna(0.0).reindex(columns=["HOLD", "RECOMMEND", "REFUSE"], fill_value=0.0)
summary.index = ["устойчивая", "риск по качеству", "деградация данных", "останов"]
print("Доля решений за неделю, %:")
display(summary)

Неделя «stable» с 2026-01-25 10:00: 168 решений раз в 60 мин, из них смен решения 3
  2026-01-25 10:00  HOLD       ЛИМС 9.6 (24 ч)    p95 11.9  установка работает
  2026-02-01 05:00  RECOMMEND  ЛИМС 8.7 (19 ч)    p95 11.6  установка работает  242000:T5 382.5→384.5
  2026-02-01 06:00  HOLD       ЛИМС 8.7 (20 ч)    p95 11.6  установка работает
  итог: {'HOLD': 167, 'RECOMMEND': 1} — {'HOLD': '99%', 'RECOMMEND': '1%'}

═══ 2026-02-01 05:00 · RECOMMEND ═══
1 Состояние   242000:T5 382.5 · 242000:F26 283.0 · 242000:F2 96 687 · 242000:P13 3.75
              ЛИМС 8.7 мг/кг (19 ч назад) · сглаженный уровень 8.54 мг/кг
              ПАК 8.4 ppm (исправен) — признак, не результат анализа
2 Проблема    вероятность нарушения спецификации 28% (порог 25%, классификатор); тяжесть режима medium (0.44)
3 Действие    242000:T5 382.5 → 384.5
4 Эффект      сера 8.54 → 7.86 мг/кг, p95 11.63 → 10.96 · риск нарушения 19% (классификатор) · тяжесть medium (0.59)
5 Проверки    sulfur_mg_kg <= 11.429; controllabl

Неделя «risk» с 2026-07-12 10:00: 168 решений раз в 60 мин, из них смен решения 14
  2026-07-12 10:00  HOLD       ЛИМС 8.1 (24 ч)    p95 9.9   установка работает
  2026-07-15 15:00  RECOMMEND  ЛИМС 10.3 (5 ч)    p95 10.8  установка работает  242000:T5 367.4→369.4
  2026-07-15 19:00  HOLD       ЛИМС 10.3 (9 ч)    p95 10.8  установка работает
  2026-07-15 22:00  RECOMMEND  ЛИМС 10.3 (12 ч)   p95 10.8  установка работает  242000:T5 371.2→373.2
  2026-07-17 05:00  HOLD       ЛИМС 11.0 (19 ч)   p95 11.1  установка работает
  2026-07-17 06:00  RECOMMEND  ЛИМС 11.0 (20 ч)   p95 11.1  установка работает  242000:T5 371.3→373.3
  2026-07-17 14:00  HOLD       ЛИМС 9.1 (4 ч)     p95 11.3  установка работает
  2026-07-17 19:00  RECOMMEND  ЛИМС 9.1 (9 ч)     p95 11.3  установка работает  242000:T5 374.2→376.2
  2026-07-17 20:00  HOLD       ЛИМС 9.1 (10 ч)    p95 11.3  установка работает
  2026-07-18 09:00  RECOMMEND  ЛИМС 10.3 (4 ч)    p95 11.5  установка работает  242000:T5 374.5→376.5
  2026-07-18

Неделя «degraded» с 2026-07-02 10:00: 168 решений раз в 60 мин, из них смен решения 12
  2026-07-02 10:00  REFUSE     ЛИМС 2.7 (348 ч)   p95 18.1  установка работает
  2026-07-02 16:00  HOLD       ЛИМС 4.1 (4 ч)     p95 9.2   установка работает
  2026-07-03 00:00  REFUSE     ЛИМС 1.1 (4 ч)     p95 8.7   установка работает
  2026-07-03 08:00  HOLD       ЛИМС 1.1 (12 ч)    p95 8.7   установка работает
  2026-07-03 09:00  REFUSE     ЛИМС 1.1 (13 ч)    p95 8.7   установка работает
  2026-07-03 12:00  HOLD       ЛИМС 1.1 (16 ч)    p95 8.7   установка работает
  2026-07-03 16:00  REFUSE     ЛИМС 1.1 (20 ч)    p95 8.7   установка работает
  2026-07-03 18:00  HOLD       ЛИМС 2.3 (5 ч)     p95 8.4   установка работает
  2026-07-04 18:00  REFUSE     ЛИМС 3.2 (8 ч)     p95 9.5   установка работает
  2026-07-04 19:00  HOLD       ЛИМС 3.2 (9 ч)     p95 9.5   установка работает
  2026-07-05 15:00  REFUSE     ЛИМС 4.8 (5 ч)     p95 9.3   установка работает
  2026-07-05 19:00  HOLD       ЛИМС 4.8 (9 ч

Неделя «shutdown» с 2026-06-21 10:00: 168 решений раз в 60 мин, из них смен решения 1
  2026-06-21 10:00  REFUSE     ЛИМС 2.7 (84 ч)    p95 12.1  установка стоит
  итог: {'REFUSE': 168} — {'REFUSE': '100%'}

═══ 2026-06-21 10:00 · REFUSE ═══
1 Состояние   242000:T5 207.0 · 242000:F26 -0.17 · 242000:P13 0.36
              ЛИМС 2.7 мг/кг (84 ч назад) · сглаженный уровень 6.33 мг/кг
              ПАК 7.9 ppm (неисправен/заморожен) — признак, не результат анализа
2 Проблема    установка не работает
3 Действие    — (отказ)
5 Проверки    sulfur_mg_kg <= 10.0; controllable tags within configured ranges; blend shares sum == 1.0 (when blend tags present); reliability.is_mode_allowed
6 Уверенность 0.00
7 Почему      Надёжной рекомендации нет: установка не работает — расход сырья ниже рабочего порога, режимные изменения не рассматриваются. Детали: 242000:F26 = -0.1714 (порог 50), 242000:T5 = 207 (порог 200)
  Катализатор цикл с 2026-04-23: данных меньше месяца
  Решение     refuse: установка не р

,HOLD,RECOMMEND,REFUSE
устойчивая,99.4,0.6,0.0
риск по качеству,62.5,37.5,0.0
деградация данных,85.7,0.0,14.3
останов,0.0,0.0,100.0


## 10. Сценарий организаторов: выросла сера в сырье

Сера в сырье растёт → растёт сера гидроочищенного ДТ → надо компенсировать. Система сравнивает два выхода:
**поднять температуру реактора** или **не трогать режим и разбавить смесь** низкосернистым компонентом.

Параметры — в ячейке 2: попробуйте `FEED_SULFUR_PCT = 1.4` или `SEASON = "winter"`.

> Исторические данные не подтверждают эффект действий, которых в них не было. Отклик режима — физическая
> модель с явными допущениями, а не наблюдение; организаторы согласились с такой постановкой 10.09.

In [12]:
from scripts import blend_scenario

args = ["--at", SCENARIO_AT, "--season", SEASON]
if FEED_SULFUR_PCT is not None:
    args += ["--feed-sulfur", str(FEED_SULFUR_PCT)]
for key, tonnes in STOCKS_T.items():
    args += ["--stock", f"{key}={tonnes}"]
blend_scenario.main(args)

Сценарий на 2026-01-27 16:00
  сера в сырье: измерено 1.0253 % → сценарий 1.25 %
  спецификация: summer

  как есть   2026-01-27 16:00  HOLD       ЛИМС 8.0 (6 ч)     p95 11.6  установка работает
  сценарий   2026-01-27 16:00  RECOMMEND  ЛИМС 8.0 (6 ч)     p95 13.5  установка работает  242000:T5 378.8→380.8

═══ 2026-01-27 16:00 · RECOMMEND ═══
1 Состояние   242000:T5 378.8 · 242000:F26 232.9 · 242000:F2 90 336 · 242000:P13 3.71
              ЛИМС 8.0 мг/кг (6 ч назад) · сглаженный уровень 8.55 мг/кг
              ПАК 6.3 ppm (исправен) — признак, не результат анализа
2 Проблема    вероятность нарушения спецификации 53% (порог 50%, оценка по интервалу)
3 Действие    242000:T5 378.8 → 380.8
4 Эффект      сера 10.43 → 9.60 мг/кг, p95 13.52 → 12.70 · риск нарушения 42% (оценка по интервалу) · тяжесть low (0.34)
5 Проверки    sulfur_mg_kg <= 11.429; controllable tags within configured ranges; blend shares sum == 1.0 (when blend tags present); reliability.is_mode_allowed; товарная смесь: сер

## 11. Сценарий: не хватает очищенного ДТ

Когда основного компонента мало, смесь собирается из керосина и газойля — и в дело вступают ограничения по
плотности и цетановому числу. Реактор здесь уже ничего не решает: смесь задают запасы.

In [13]:
blend_scenario.main(["--at", SCENARIO_AT, "--stock", "hydrotreated_diesel=300"])

Сценарий на 2026-01-27 16:00
  запасы: {'hydrotreated_diesel': 300.0}

  как есть   2026-01-27 16:00  HOLD       ЛИМС 8.0 (6 ч)     p95 11.6  установка работает
  сценарий   2026-01-27 16:00  HOLD       ЛИМС 8.0 (6 ч)     p95 11.6  установка работает

═══ 2026-01-27 16:00 · HOLD ═══
1 Состояние   242000:T5 378.8 · 242000:F26 232.9 · 242000:F2 90 336 · 242000:P13 3.71
              ЛИМС 8.0 мг/кг (6 ч назад) · сглаженный уровень 8.55 мг/кг
              ПАК 6.3 ppm (исправен) — признак, не результат анализа
2 Проблема    признаков проблемы нет: прогноз серы 8.55 мг/кг, тяжесть режима low
3 Действие    режим не менять
4 Эффект      сера 8.55 → 8.55 мг/кг, p95 11.65 → 11.65 · риск нарушения 4% (классификатор) · тяжесть low (0.20)
5 Проверки    sulfur_mg_kg <= 10.0; controllable tags within configured ranges; blend shares sum == 1.0 (when blend tags present); reliability.is_mode_allowed; товарная смесь: сера ≤ 10 мг/кг; товарная смесь: T95 ≤ 360 °C; товарная смесь: цетановое число ≥ 51 (ле

## 12. Воспроизводимость

Одинаковое состояние — одинаковое решение и одинаковые числа. Два независимых прогона одного момента:

In [14]:
def trace(at: str) -> str:
    with contextlib.redirect_stdout(io.StringIO()):
        rec = run_real.main(["--at", at])[0]
    return json.dumps(rec.model_dump(mode="json"), sort_keys=True, ensure_ascii=False)

first, second = trace("2026-07-16T14:00"), trace("2026-07-16T14:00")
print(f"трасса {len(first):,} символов; прогоны совпадают побайтово: {first == second}")
assert first == second

трасса 20,918 символов; прогоны совпадают побайтово: True


## 13. Допущения

Каждое допущение записано с причиной в `ASSUMPTIONS.md`.

In [15]:
from IPython.display import Markdown

assumptions = ROOT / "ASSUMPTIONS.md"
if assumptions.exists():
    display(Markdown(assumptions.read_text(encoding="utf-8")))
else:
    print("ASSUMPTIONS.md не найден: .gitignore репозитория пока исключает *.md")

# ASSUMPTIONS

Все допущения проекта с причиной (правило 6 CLAUDE.md; оцениваемый артефакт:
«все допущения должны быть явно описаны»). Пополняется в момент, когда допущение
принимается. ❓ — можно снять вопросом организаторам.

> Файл пока не попадает в git: `.gitignore` репозитория исключает `*.md`. Нужна правка
> Кирилла, иначе артефакт не войдёт в сдачу.

---

## Данные

| Допущение | Причина | Где в коде |
|---|---|---|
| ❓ Значения `307`, `0`, `10` в телеметрии 24-2000 — заглушки пропусков, а не измерения | NaN в данных ноль; `307` встречается одновременно в давлениях, температурах и расходах. Вывод по повторяемости, не по документации | `scripts/realdata.py` → `SENTINELS_242000` |
| Установка работает, если `242000:F26` > 50 и `242000:T5` > 200 | Остановы записаны почти-нулевыми float (`F26` ≈ −0.14, `T5` ≈ 139), точечная чистка их не ловит | `RUNNING_MIN` |
| Для статистик берутся только «чистые» рабочие строки: все четыре рычага есть и выше порогов (`F2` > 10 000, `P13` > 1.0) | Иначе процентили тянутся к нулю остановов (p01 `T5` = 5 °C) | `CLEAN_OPERATING_MIN` |
| ✅ Время в ЛИМС — время отбора; результат доступен через **4 ч** | **Подтверждено организаторами 10.09.2026:** «Метка времени это момент отбора пробы. До публикации анализа в системе проходит до 4 часов». Совпало с нашим допущением дословно | `LIMS_DELAY` |
| ПАК считается замороженным после 36 одинаковых показаний подряд (6 ч) и устаревшим после 2 ч | Шумный сигнал в ppm не повторяется до последнего знака на исправном приборе | `PAK_FROZEN_RUN`, `PAK_MAX_AGE` |
| Лабораторные результаты выше 50 мг/кг помечаются как выбросы и не участвуют в подгонке, но остаются в таблице | Все три (2120 и 107 — 23.04.2024, 120 — 24.07.2025) пришлись на первые 3–64 ч после пуска установки: похоже на пусковой режим, а не на ошибку единиц. Окончательное решение за командой | `OUTLIER_ABOVE_MG_KG`, `models/labels.parquet` |
| Разбиение по времени на 2025-09-12 14:30; всё обучение — только до этой границы | CLAUDE.md §6.2 | `SPLIT_AT` |
| **Неделя «деградации» перенесена на 02.07.2026** (с 19.09); прежняя неделя 21.06.2026 оставлена отдельным сценарием «останов» | Прежняя неделя оказалась остановом установки — это самый слабый вид отказа. С 02.07 установка работает всю неделю, а данные отказывают: последний анализ ЛИМС 348 ч назад, анализатор неисправен 58 % недели. Система отказывает, пока нет источника качества; решает на запасной оценке, пока не работает только анализатор; и снова отказывает, когда свежий катализатор уводит реактор ниже виденного. Неделя найдена перебором отложенного периода по доле часов «установка работает + анализ старше 30 ч + анализатор неисправен» | `DEMO_WEEKS` |
| ✅ Справочник тегов 24-2000 заменён на исправленный (`теги АВТ_24-2000.xlsx`, 16.09.2026): впервые есть физическая величина и единица у каждого тега | Подтверждает вывод аудита: старая колонка описаний была перепутана. Описания изменились у `T5`, `T6`, `P8`, `F9`, `T11`, `P13`, `F14`, `F15`, `F19`, `Q20`, `Q21`, `F26`. Список рычагов, который организаторы назвали в чате (`P8`, `T11`, `F19`), процитирован из **старой** таблицы: по исправленной это перепад давления, температура после Р-202 и расход бензина | `TAG_UNITS` |
| Единицы КИП больше не выводятся из диапазонов — они взяты из справочника | Раньше каждая единица была нашей инференцией (аудит §6.1). Прозрачность единиц оценивается | `TAG_UNITS`, `LEVERS` |
| `Q21` — поточный анализатор серы в гидроочищенном ДТ, внутри файла телеметрии | Медиана 8.43 ppm против 8.45 у выгрузки ПАК, средняя разница 0.84. Это тот же сигнал на 10-минутной сетке; вошёл в признаки классификатора | `CONTEXT_TAGS` |
| `Q20` (медиана 8150 ppm = 0.82 % масс.) по шкале похож на серу **в сырье**, но лабораторию сырья не отслеживает: Спирмен −0.03 при 122 парах | Масштаб подходит, поведение нет. Как признак допустим, как источник истины — нет | — |
| **Поправка к аудиту.** ПАК и ЛИМС по сере связаны, но только на **нулевом лаге**: ранговая корреляция 2-часового среднего ПАК с результатом ЛИМС +0.41 на обучении и +0.31 на отложенном периоде. На окнах 6 и 24 ч слабее, на лагах от 2 ч исчезает | Вывод аудита «два ряда по одному потоку не связаны ни на одном лаге» получен по Пирсону на сырых значениях. Три выброса выше 50 мг/кг из 1 067 переводят корреляцию обучающего периода с **+0.33 в −0.002** — это артефакт выбросов, а не отсутствие связи; ранговая статистика к ним устойчива. Связь устойчива во времени: AUC по полугодиям 0.58–0.81, ни одного полугодия ниже 0.5 | `scripts/analysis/breach_classifier.py` → `models/breach_classifier.json` |

## Агент качества (`QualityAgentBaseline`, модель `ewma_v0`)

| Допущение | Причина |
|---|---|
| Прогноз серы — экспоненциальное сглаживание доступных результатов ЛИМС, α = 0.1 | Соседние анализы почти не связаны (автокорреляция +0.09 на обучении). «Как вчера» хуже медианы (MAE 1.73 против 1.52 на отложенном периоде). Сглаживание, выбранное на обучении, лучше обоих: MAE 1.46 |
| Интервал p05–p95 = прогноз + квантили ошибки сглаживания на обучении (−3.47 / +3.10 мг/кг); расширяется как √(возраст / 24 ч) при устаревшем анализе | Покрытие на отложенном периоде 90.4 % при цели 90 % |
| Влияние рычагов (мг/кг на единицу): `T5` −0.35, `F26` +0.05, `P13` −1.75; отношения `F2/F26` −0.025 | **Знаки — физика процесса, величины выбраны, не обучены.** Для давления взят локальный наклон степенной зависимости сера ~ P^−0.8 при 3.92 МПа и 8.5 мг/кг. Проверка предсказуемости сигнала не нашла (R² на отложенном −0.005), поэтому величины остаются допущениями |
| Уверенность 0.8 × 1/(1 + возраст/72 ч), × 0.5 при неисправном ПАК, × 0.25 без анализа | Прозрачное правило, не калибровка |
| Сера оценивается в **гидроочищенном** ДТ, а не в товарном | Цепочка данных заканчивается на гидроочистке; не автоматически консервативно |
| Обученная модель серы **не используется**: агент качества остаётся на сглаживании | Проверка предсказуемости (§12.2): ридж на средних по окнам 2/6/24 ч, лаги 0–24 ч, три набора признаков (рычаги; + справочные теги; + ПАК и `Q21`) — R² на отложенном периоде выбранного по CV варианта **−0.005**; градиентный бустинг — R² обучения 0.91, отложенного −0.04 (переобучение). Модель, которая не лучше сглаживания, не выдаётся за прогноз (`models/predictability.json`) |
| Отклик на рычаги **мультипликативный** (с 18.09): сера = уровень × exp(Σ чувствительность × Δрычаг / 8.5). Локальный наклон на уровне 8.5 мг/кг равен заявленной чувствительности, конечный шаг чуть меньше: 2 °C `T5` дают −0.67, а не −0.70 | Линейная модель могла увести прогноз в отрицательную серу и не отвечала на серу в сырье. Температура входит экспонентой, как в кинетике Аррениуса; для давления и расхода экспонента совпадает со степенным законом на шагах в несколько процентов. Исходы демо-недель после замены не изменились |
| Сценарий «изменилась сера в сырье»: уровень серы продукта умножается на (сценарий / измерено)^1 | Кинетика первого порядка — сера продукта пропорциональна сере сырья. Измеренная сера сырья — последний опубликованный анализ ЛИМС «Гидроочистка, т.о. 1» (132 анализа за 3.5 года, поэтому часто недельной давности); она служит только опорой для отношения |
| В сценарии с изменённым сырьём классификатор не применяется | Он читает анализатор, который видит реальное сырьё, а не гипотетическое. Риск берётся из сдвинутого интервала |
| Величины «что если» остаются допущениями, а не коэффициентами модели | В регрессии только на рычагах знак `F26` совпал с физикой, а `T5` и `P13` — нет; при R² ≈ 0 это шум, а не зависимость (CLAUDE.md §11: неверный знак не выкатывается). В классификаторе, где сигнал есть, все три знака физичные |
| Классификатор «анализ выше 10 мг/кг» — **оценка «сейчас», а не прогноз**. Горизонт 0 ч: AUC на отложенном периоде **0.724** (бутстрап 0.660–0.784), PR-AUC 0.35 при базовой доле 0.17, Brier skill +0.046. На горизонтах 2–24 ч AUC 0.39–0.57, сигнала нет | Все четыре критерия приёмки, заданные **до** запуска, выполнены: AUC ≥ 0.60, нижняя граница бутстрапа > 0.5, Brier skill > 0, лучше текущей вероятности агента (AUC 0.513). Выбор горизонта, набора признаков и регуляризации — только по CV внутри обучающего периода; отложенный период оценён один раз |
| Сигнал несёт ПАК, а не рычаги: 2-часовое среднее ПАК в одиночку даёт AUC 0.68 на отложенном периоде (CV обучения 0.72), рычаги в одиночку — CV 0.47 | Физически это и ожидаемо: анализатор измеряет ту же величину в тот же момент. Поэтому классификатор применим только при исправном ПАК; при замороженном или устаревшем анализаторе остаётся интервальная оценка |
| Три выброса выше 50 мг/кг входят в классификатор как обычные нарушения (`1`) | Для бинарной задачи это просто «выше 10», величина не искажает подгонку. В регрессию и сглаживание они по-прежнему не входят. Решение Романа, 17.09 |
| После перехода на исправленный справочник `T5` стал **самым сильным** коэффициентом классификатора: −0.73, знак физически верный. У `P13` −0.24, тоже верный | В регрессии знак `T5` выходил обратным. CLAUDE.md §11 выполняется без принуждения монотонности. Порог триггера пересчитан на 0.2512 |

## Агент надёжности (`ReliabilityAgentBaseline`)

| Допущение | Причина |
|---|---|
| Рабочий диапазон рычагов = p01–p99 обучающего периода на чистых рабочих строках | Модельная граница, **не заводской лимит**: в пакете нет паспортных диапазонов |
| Прокси дезактивации катализатора: насколько `T5` выше медианы обучения при том же `F26` (полосы по 10 м³/ч) | Нет возраста катализатора и температур по слоям. Медиана `T5` монотонно растёт с `F26` (349 → 374 °C), то есть связь в данных физически осмысленна |
| Индекс тяжести = 0.5·температура + 0.3·производительность + 0.2·размер шага; классы low < 0.4 ≤ medium < 0.7 ≤ high; режим не допускается при ≥ 0.85 | Веса и пороги выбраны, не обучены |
| **Допуск в один шаг рычага у границы рабочего диапазона** для текущего режима; сдвинутый рычаг обязан оказаться внутри диапазона | Без допуска решение дребезжало: установка у края диапазона каждый час то получала принудительную рекомендацию, то отказ. В неделю риска смен решения стало 13 вместо 24, исчезли искусственные рекомендации «сбросить давление». Правило для сдвинутых рычагов совпадает с проверкой белого списка Кирилла, поэтому два агента не противоречат друг другу |
| Причина отказа «вне диапазона» называется словами агента надёжности: какой рычаг, где он и какой диапазон | Жёсткая проверка сообщала только «reliability agent marked mode as not allowed». Теперь карточка пишет, например, «242000:T5 = 329.3 вне рабочего диапазона [347; 387.9] больше чем на шаг рычага» |
| **Ресурс катализатора** — по избытку `T5` над медианой обучения при той же загрузке: рост при постоянной загрузке — классический признак дезактивации | В данных видна «пила»: +1.1 °C/мес в каждом цикле и сбросы на заменах. Паспортных данных о катализаторе в пакете нет |
| Замена катализатора распознаётся по данным: пуск, после которого избыток температуры падает не меньше чем на 8 °C (три недели работы до и после) | Найдены 17.04.2024 (обучение) и 23.04.2026 (отложенный период, останов 186 ч, падение 29.5 °C). Обе даты совпадают с реальными длительными остановами |
| **Уровень замены +13.56 °C** — избыток, при котором катализатор меняли в обучающем периоде | Не паспортная граница, а наблюдаемая практика завода, выведенная только из обучения. В 2026 году катализатор сменили позже, при +16.7 °C |
| Дрейф — линейный тренд за последний год цикла; первые 90 дней свежего катализатора не экстраполируются; текущий износ — худшая из двух оценок: по тренду и по последним двум неделям | Свежий катализатор теряет активность быстро, и экстраполяция начала цикла давала «+21 °C/мес». Год сглаживает сезонность сырья. **Правило «худшей оценки» добавлено после того, как ретроспектива на отложенном периоде показала запаздывание прогноза** — поэтому числа ниже уже не чистая проверка. Направление выбрано из соображений безопасности: предупреждение об износе, пришедшее поздно, хуже раннего |
| Ретроспектива прогноза замены против фактической 23.04.2026: за 3 недели — «уровень замены достигнут»; за 7 недель — ошибка +4 недели; за полгода — от +6 до +114 дней | Для предупреждения на горизонте 1–2 месяца годится, для полугодового планирования — нет. Линейный тренд не видит ускорения износа в конце цикла |
| Через три недели после любого останова дольше 72 ч цикл катализатора не оценивается | Что пуск был заменой катализатора, видно только по падению температуры за три недели после. Без этого правила статус подсматривал бы в будущее |
| Ресурс катализатора — информация и мягкое ограничение «не повышать T5» за 60 дней до уровня замены; индекс тяжести не меняет | Сегодняшний избыток температуры уже входит в индекс через температурную составляющую; тренд и срок — новая информация, а не повод менять решения задним числом |

## Оптимизатор и оркестратор

| Допущение | Причина |
|---|---|
| Шаги рычагов за один цикл: `T5` 2 °C, `F26` 5 м³/ч, `F2` 1 000 нм³/ч, `P13` 0.05 МПа | Малые изменения около текущего режима (CLAUDE.md §11) |
| **Набор рычагов изменён 18.09.2026:** `F15` убран, `P13` добавлен, `F26` переподписан | `F15` по исправленному справочнику — «расход сырья объёмный», но 3 400 м³/ч против массового расхода 220 т/ч дают плотность 0.065: это не он, а квенч — это `F14`. Тег остаётся неопознанным и рычагом быть не может. `P13` (давление на входе Р-202) организаторы назвали управляющей переменной, и CLAUDE.md §11 требует, чтобы рост парциального давления водорода снижал серу. `F26` — расход **гидроочищенного ДТ в цех №8**, а не сырья; рычагом остаётся, потому что `F9`/`F26` = 0.850 т/м³ (плотность дизеля) и корреляция 0.9999 — один поток в двух единицах |
| Давление — сознательно слабый рычаг: один шаг даёт −0.09 мг/кг против −0.67 у `T5` | Весь исторический размах `P13` — 0.42 МПа (p01 3.61, p99 4.03). Делать его сильным означало бы выдумать чувствительность, которой в данных нет |
| Если единственный признак проблемы — тяжесть режима, рассматриваются только шаги, которые её **не увеличивают** | Рост `T5` снижает серу и потому выигрывает по score, но отвечать на «оборудование работает напряжённо» действием, которое напряжение увеличивает, — не ответ. Дефект был виден на часовом темпе: 01.02.2026 03:00 система поднимала `T5` 382.4 → 384.4 при тяжести 0.45 → 0.60. Фильтр применяется до сравнения, а не через вес (CLAUDE.md §2 правило 4) |
| Темп выдачи рекомендаций — **раз в час**, горизонт оценки — **3 часа** | Требование организаторов от 10.09.2026: «Шаг выдачи рекомендаций от 15 минут до 1 часа. Горизонт прогноза от 0 до 3-х часов», запаздывание тоже 0–3 ч. Внутри горизонта две шкалы, и трасса их различает: риск — оценка на 0–2 ч (классификатор), интервал — по-прежнему суточный разброс лаборатория-к-лаборатории. Суточный разброс на трёх часах завышает неопределённость, то есть расширяет p95 и склоняет к отказу — сторона безопасная | `HORIZON_MINUTES`, `DEFAULT_EVERY_MINUTES` |
| Порог устаревания ЛИМС поднят с 24 до 30 ч — **правка в секции Кирилла**, нужно согласовать | Проба берётся раз в сутки около 10:00 и публикуется через 4 ч, поэтому порог 24 ч гарантированно давал отказ каждый день с 11:00 до 14:00: на часовом темпе это 3 отказа из 24 в полностью нормальном режиме. Правильное решение — при устаревшей лаборатории опираться на исправный анализатор, а не отказывать; порог снимает симптом до этой переделки | `configs/constraints.yaml` |
| Производительность уменьшает score (меньше — лучше), риски и энергия увеличивают | Исправление знака в исходном `_score` |
| При равенстве score выбирается «ничего не менять» | Изменение, которое ничего не даёт, не стоит действия оператора |
| Если установка стоит, решение — отказ, до любых других проверок | Режимные рекомендации для остановленной установки бессмысленны |
| Кандидат двигает **один** рычаг за раз: удержание + каждый рычаг на ±шаг, 9 вариантов (`decision.max_levers_per_action: 1`) | Оператор выполняет и проверяет одно изменение за раз; карточка, двигающая четыре уставки, не может объяснить, какая сработала |
| Изменение рекомендуется **только при признаке проблемы**: последний анализ ЛИМС > 10 мг/кг; прогнозная вероятность нарушения ≥ 50 %; тяжесть режима medium/high; текущий режим не проходит ограничения. Без признака — «режим не менять» | Режим в пределах спецификации не меняют ради небольшого улучшения на бумаге. Без этого правила устойчивая неделя давала рекомендацию 7 дней из 7 |
| Порог удержания `decision.hold_margin` = 5 единиц score: изменение должно быть лучше удержания хотя бы на столько (≈ 5 п.п. вероятности нарушения при весе 100) | Выбран, не обучен. Следствие: 12.07.2026 анализ 11.6 — признак есть, но лучший шаг даёт лишь 3.4 → «не менять», и карточка пишет почему |
| Если текущий режим не проходит ограничения, лучший допустимый шаг рекомендуется без порога удержания; если допустимых нет — отказ «нужно решение человека» | Возврат в допустимую область важнее гистерезиса; когда вернуть нечем, решение за человеком |

## Блендинг (`BlendingAgent`, с 18.09)

Схема организаторов «Блендинг из резервуаров» и спецификация товарного резервуара:
сера ≤ 10 мг/кг, T95 ≤ 360 °C, цетановое число ≥ 51 (зимой ≥ 49), плотность 820–845
кг/м³ (зимой 800–845). Свойства компонентов считает `scripts/analysis/blend_components.py`
→ `models/blend_components.json`; сценарные параметры — `configs/blending.yaml`.

| Допущение | Причина |
|---|---|
| **Компонент 1 — гидроочищенный ДТ** — полностью из ЛИМС «Гидроочистка, т.о. 2»: плотность 836.1, T95 347 °C, сера 8.6 мг/кг, цетановое число **53.75** (42 замера) | Цетановое число в данных есть, хотя замеров мало. В смесь сера идёт не медианой, а средним прогнозом серы для выбранного режима — смесь отвечает на решение по гидроочистке |
| **Керосин** представлен фракцией «АВТ, т.о. 2», **газойль** — фракцией «АВТ, т.о. 1» | В пакете нет своих точек отбора для этих компонентов. АВТ т.о. 2 — самая лёгкая фракция (T95 296 °C, температура помутнения −23 °C), АВТ т.о. 1 — самая тяжёлая (плотность 882) |
| Обе фракции берутся в **гидроочищенном** виде: к их плотности и разгонке прибавлен сдвиг, который гидроочистка даёт на дизельной фракции (т.о. 1 → т.о. 2): плотность −11.3, T50 −5 °C, T95 −6 °C | Прямогонная фракция в продукт с нормой 10 мг/кг не годится: сера сырья гидроочистки 0.95 % = 9 500 мг/кг. Сдвиг измерен, а не выдуман |
| **Цетановое число керосина (52.1) и газойля (48.5) рассчитано по ASTM D976** из плотности и T50 | Стандартный двухпараметрический цетановый индекс. **Проверен на 37 парах замеров гидроочищенного ДТ**, где цетан измерялся: средняя ошибка 1.7 единицы, смещение −1.1 (формула занижает), медиана 53.3 против измеренных 53.8. Это в пределах воспроизводимости самого стандарта. Смещение не поправляем: заниженный цетан ведёт к лишней присадке, то есть ошибка в безопасную сторону |
| Сера керосина **5 мг/кг**, газойля **8 мг/кг** — **единственные выдуманные свойства** компонентов | Сернистых анализов по этим точкам нет. Обоснование: чтобы компонент годился для продукта с нормой 10 мг/кг, он должен быть гидроочищенной ультранизкосернистой маркой; а в лёгкой фракции сера удаляется легче, потому что трудные соединения (алкилдибензотиофены) концентрируются в тяжёлом конце. Отсюда керосин ниже газойля |
| 53 нефизичных значения во всём файле ЛИМС отброшены до расчёта: `D15` вне 700–1000 (1), `50%.T` вне 100–450 (5), `95%.T` вне 150–500 (47), `CetaneNumber` вне 20–80 (0) | В файле есть нули и подобные ошибки ввода. Их считаем и показываем, не подменяем |
| Смешение: сера — линейно по массе, плотность — по объёму (1/ρ = Σ w/ρ); T95 и цетан — линейно по объёму | Сера и плотность — точно. T95 и цетан на НПЗ смешивают по индексам, линейность — **допущение**; организаторы разрешили «достаточно простую, в т.ч. линейную» модель |
| Цены всех компонентов равны | Ценовых данных в пакете нет. Выдуманная разница стала бы фикцией, на которой оптимизатор строил бы выводы. Экономику задаёт только присадка |
| Присадка: цена **в 100 раз** выше тонны ДТ, доза до 30 кг/т (3 %) | Дословно из ответа организаторов |
| Отклик присадки насыщается: +10·(1 − exp(−доза / 2.5 кг/т)) единиц ЦЧ | Порядок величин для цетаноповышающих присадок: несколько единиц на 1 кг/т с насыщением. Линейный отклик при разрешённых 3 % дал бы +100 единиц, что физически невозможно. Параметры — допущения |
| Вторая присадка со схемы организаторов выключена | Её свойств в пакете нет. Слот в конфиге оставлен, включается одной строкой |
| Порядок выбора среди допустимых смесей: 1) дешевле; 2) больше собственного продукта — менять как можно меньше; 3) больше запас до ближайшей границы | При равных ценах без правила 2 оптимизатор разбавлял бы продукт без нужды. Ограничения — фильтр, а не вес (CLAUDE.md §2 правило 4) |
| Размер партии 1 000 т; запасы: ДТ 1 200 т, керосин 300 т, газойль 400 т | Данных о запасах нет. Значения выбраны так, чтобы в обычном режиме смесь была 100 % ДТ, а сценарии нехватки задавались параметром `--stock` |
| Смесь считается для **выбранного** режима, без совместной оптимизации с рычагами | Решение Романа от 18.09: сначала последовательная схема, совместная — если хватит времени. Сценарий всё равно показывает оба пути — «реактор» и «резервуар» |
| Если ни один шаг режима не исправляет серу гидроочистки, решение по режиму — отказ, но смесь всё равно считается | Резервуар может спасти товарный продукт, когда реактор не может. Карточка это показывает |
| **Серный бюджет**: агент смеси сообщает гидроочистке, сколько серы в ДТ резервуар доведёт до нормы, и жёсткая проверка режима идёт по нему, а не по 10 мг/кг | Норма 10 мг/кг по ТЗ относится к товарной смеси, а не к промежуточному продукту. Это настоящий обмен между агентами: гидроочистка сама не может знать, что есть в резервуарах |
| Бюджет считается **без присадки**, с **не больше 30 % разбавителя** и с **запасом 0.5 мг/кг** до нормы смеси; при текущих запасах — 11.43 мг/кг (70 % ДТ + 30 % керосина) | Без ограничения на разбавитель арифметика давала 16 мг/кг через смесь, где ДТ всего 30 %, — но установка производит продукт непрерывно и спрятать остальное не может. Присадка стоит в 100 раз дороже ДТ, покупать ею бюджет нельзя. Все три величины — допущения, задаются в `configs/blending.yaml` |
| Бюджет может только **ослабить** норму 10 мг/кг для гидроочищенного ДТ, но не ужесточить | Без агента смеси режим и так проверялся по 10; бюджет лишь добавляет запас, который даёт резервуар. Иначе запас 0.5 мг/кг без разбавителя превратил бы норму в 9.5 |
| **Резерв экономии** («T5 можно снизить до …») — только информация на карточке, не рекомендация, и только когда режим удерживается без признаков проблемы | Организаторы: чем больше серы в гидроочищенном ДТ, тем дешевле его производство. Превращать это в рекомендацию значило бы изменить смысл устойчивой недели — вопрос оставлен Роману; по умолчанию выбран осторожный вариант. Отклик — модель «что если» агента качества, то есть допущение |
| **Проверки смеси живут в `agents/blending.py`, а не в `safety/constraints.py`** | По CLAUDE.md §9 все жёсткие ограничения в одном месте, но `safety/` — файлы Кирилла. Переносить — по согласию |


## Где что смотреть

| Критерий | Разделы |
|---|---|
| Работа с данными | 1, 2 — заглушки, остановы, задержка ЛИМС, разбиение по времени |
| Качество прогноза | 3, 4, 5 — сглаживание, честный отрицательный результат, классификатор с критериями до запуска |
| Мультиагентная архитектура | 8, 9 — пять агентов, обмен через состояние и трассу |
| Оптимизация и безопасность | 9, 10, 11 — жёсткий фильтр до сравнения, отказ с причиной |
| Надёжность | 6 — тяжесть режима, модельный диапазон |
| Объяснимость | 8 — карточка: причина → действие → эффект → проверки |
| Воспроизводимость | начало ноутбука, 12 — кэш из сырых файлов, побайтово одинаковая трасса |
| Демонстрация | 9, 10, 11 |

In [16]:
print(f"весь пайплайн: {time.time() - T0:.0f} с")

весь пайплайн: 63 с
